# ⚡ PySpark Bootcamp — Zero to Hero (for Databricks)

A complete, hands-on tour of **PySpark** — the Python API for **Apache Spark**,
the distributed engine that powers Databricks. Every cell is self-contained: the
notebook **builds its own sample data**, so there's nothing to upload.

**How to use it**
1. Import into Databricks: *Workspace → Import → File* → this `.ipynb`.
2. Attach to any cluster / serverless compute.
3. Run cells **top to bottom** — later sections use the DataFrames created near
   the top. Read, run, tweak, re-run.

**Covered:** Spark architecture & lazy evaluation, creating & inspecting
DataFrames, `select`/`filter`/`withColumn`, nulls, sorting, `groupBy`/`agg`, all
**joins**, `union`, **window functions**, string & date functions, complex types
(`array`/`struct`/`explode`), **UDFs** & pandas UDFs, **Spark SQL** & temp views,
**Delta Lake** (write/read/time-travel/`MERGE`), caching & performance, and
pandas interop.

## 1 · How Spark works (the mental model)

Spark runs your job across a **cluster**: a **driver** (your notebook) plans the
work and **executors** (many machines) do it in parallel on partitions of the
data.

Two ideas you must internalize:

- **Transformations are lazy.** `select`, `filter`, `withColumn`, `join`,
  `groupBy` just *describe* a computation — nothing runs yet. Spark builds a plan
  (a DAG) and optimizes it (Catalyst).
- **Actions trigger execution.** `show`, `count`, `collect`, `toPandas`,
  `write` force Spark to actually run the plan and return/persist results.

On Databricks the **`spark`** session is already created for you. (The cell below
also works outside Databricks if PySpark is installed.)

In [ ]:
try:
    spark                                   # pre-injected on Databricks
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("pyspark-bootcamp").getOrCreate()

print("Spark version:", spark.version)

## 2 · Create the sample data

We build a small fictional retailer with `spark.createDataFrame` from Python
lists — no files needed. Spark **infers** the schema (types) from the data.

In [ ]:
from pyspark.sql import functions as F, Window
from datetime import date, datetime

customers = spark.createDataFrame([
    (1, "Ava Smith",   "US", "ava@x.com",    date(2023,1,15)),
    (2, "Liam Patel",  "GB", "liam@x.com",   date(2023,3,2)),
    (3, "Mei Kim",     "de", None,           date(2023,5,20)),
    (4, "Noah Garcia", "US", "noah@x.com",   date(2023,6,11)),
    (5, "Olivia Rossi","IT", "olivia@x.com", date(2023,8,9)),
    (6, "Raj Haddad",  "in", None,           date(2023,9,30)),
    (7, "Sofia Silva", "br", "sofia@x.com",  date(2024,1,5)),
    (8, "Chen Wu",     "US", "chen@x.com",   date(2024,2,18)),
], ["customer_id", "name", "country", "email", "signup_date"])

products = spark.createDataFrame([
    (101,"Wireless Mouse","Electronics",24.99),
    (102,"Mechanical Keyboard","Electronics",79.50),
    (103,"Novel","Books",14.00),
    (104,"Coffee Mug","Home",9.75),
    (105,"Desk Lamp","Home",39.90),
    (106,"Lego Set","Toys",59.99),
    (107,"Water Bottle","Home",12.50),
    (108,"Headphones","Electronics",129.00),
], ["product_id","product_name","category","unit_price"])

orders = spark.createDataFrame([
    (1001,1,datetime(2024,1,10,9,15),"completed",104.49),
    (1002,2,datetime(2024,1,12,14,30),"completed",79.50),
    (1003,1,datetime(2024,2,1,11,5),"returned",14.00),
    (1004,3,datetime(2024,2,14,16,45),"completed",168.90),
    (1005,4,datetime(2024,3,3,8,20),"completed",59.99),
    (1006,5,datetime(2024,3,19,19,10),"cancelled",39.90),
    (1007,2,datetime(2024,4,7,12,0),"completed",141.50),
    (1008,1,datetime(2024,4,22,10,30),"completed",22.25),
    (1009,7,datetime(2024,5,5,13,50),"completed",129.00),
    (1010,8,datetime(2024,5,30,17,25),"returned",24.99),
    (1011,4,datetime(2024,6,11,15,40),"completed",92.49),
    (1012,3,datetime(2024,6,28,9,5),"completed",49.65),
], ["order_id","customer_id","order_ts","status","amount"])

employees = spark.createDataFrame([
    (1,"Dana Lee",None,"Executive",185000),
    (2,"Marcus Cole",1,"Sales",98000),
    (3,"Priya Nair",1,"Engineering",122000),
    (4,"Tom Becker",2,"Sales",72000),
    (5,"Ines Dubois",2,"Sales",69000),
    (6,"Sara Ahmed",3,"Engineering",95000),
    (7,"Leo Marconi",3,"Engineering",101000),
    (8,"Grace Park",3,"Engineering",88000),
], ["employee_id","name","manager_id","department","salary"])

order_items = spark.createDataFrame([
    (1001,101,2),(1001,104,1),(1004,108,1),(1004,105,1),(1005,106,1),
    (1007,108,1),(1007,107,1),(1009,108,1),(1011,102,1),(1012,105,1),
], ["order_id","product_id","quantity"])

print("DataFrames created.")

## 3 · Inspecting a DataFrame

`show()` prints rows (an **action**); `printSchema()` shows column types;
`count()` counts rows; `columns`/`dtypes` describe structure. In Databricks,
**`display(df)`** renders a rich, interactive, chartable table — use it instead
of `show()` there.

In [ ]:
orders.show(5, truncate=False)
orders.printSchema()
print("rows:", orders.count(), "| columns:", orders.columns)
# In a Databricks notebook you'd write:  display(orders)

## 4 · Selecting columns

`select` chooses columns. Reference a column as a string, as `F.col("x")`, or
with `df.x`. `alias` renames; `F.expr` runs a SQL expression.

In [ ]:
orders.select("order_id", "amount").show(3)

orders.select(
    F.col("order_id"),
    (F.col("amount") * 1.2).alias("amount_with_tax"),
    F.expr("upper(status) AS status_uc"),
).show(3)

## 5 · Filtering rows

`filter` (alias `where`) keeps matching rows. Combine conditions with `&`, `|`,
`~` (each wrapped in parentheses). Handy: `isin`, `between`, `like`, `isNull`.

In [ ]:
orders.filter((F.col("amount") >= 100) & (F.col("status") == "completed")).show()

customers.where(F.col("country").isin("US", "GB")).select("name","country").show()

customers.filter(F.col("email").isNull()).select("customer_id","name").show()

## 6 · Adding & transforming columns

`withColumn` adds/replaces a column. `F.when(...).otherwise(...)` is SQL's
CASE. `cast` changes types.

In [ ]:
enriched = (orders
    .withColumn("amount_tier",
        F.when(F.col("amount") >= 100, "big").otherwise("small"))
    .withColumn("is_completed", F.col("status") == "completed")
    .withColumn("order_date", F.to_date("order_ts")))
enriched.select("order_id","amount","amount_tier","is_completed","order_date").show(5)

## 7 · Handling nulls

`df.na.fill(...)` fills, `df.na.drop(...)` drops, `F.coalesce` picks the first
non-null. Some customers have no email.

In [ ]:
customers.na.fill({"email": "no-email"}).select("name","email").show()
print("rows with an email:", customers.na.drop(subset=["email"]).count())
customers.select("name", F.coalesce("email", F.lit("UNKNOWN")).alias("email")).show(3)

## 8 · Sorting

`orderBy` (alias `sort`) sorts; use `F.desc(...)` / `F.asc(...)` or
`.desc()`/`.asc()` on a column. `limit` caps rows.

In [ ]:
orders.orderBy(F.desc("amount")).select("order_id","amount","status").show(5)

## 9 · Aggregation & `groupBy`

Aggregate the whole DataFrame with `agg`, or per group with `groupBy().agg()`.
Aggregate functions live in `pyspark.sql.functions` (`F.sum`, `F.avg`,
`F.count`, `F.countDistinct`, `F.min`, `F.max`).

In [ ]:
orders.agg(
    F.count("*").alias("n"),
    F.round(F.avg("amount"), 2).alias("avg_amount"),
    F.max("amount").alias("max_amount"),
).show()

(orders.groupBy("status")
    .agg(F.count("*").alias("n_orders"),
         F.round(F.sum("amount"), 2).alias("revenue"),
         F.round(F.avg("amount"), 2).alias("avg_amount"))
    .orderBy(F.desc("revenue"))
    .show())

## 10 · Joins

`df1.join(df2, on, how)` combines tables. If the key column has the same name in
both, pass it as a string to avoid duplicate columns. `how` = `inner` (default),
`left`, `right`, `outer`, plus Spark's `left_semi` and `left_anti`.

In [ ]:
# Inner join: each order with its customer
(orders.join(customers, "customer_id", "inner")
    .select("order_id", "name", "amount")
    .show(5))

# Left join + aggregate: order count per customer (non-buyers show 0)
(customers.join(orders, "customer_id", "left")
    .groupBy("customer_id", "name")
    .agg(F.count("order_id").alias("n_orders"))
    .orderBy("n_orders", "name")
    .show())

In [ ]:
# SEMI / ANTI joins filter the LEFT table by existence, returning only left cols
print("Customers who ordered (semi):")
customers.join(orders, "customer_id", "left_semi").select("customer_id","name").show()

print("Customers who never ordered (anti):")
customers.join(orders, "customer_id", "left_anti").select("customer_id","name").show()

In [ ]:
# Self-join: employee -> manager (join a DataFrame to itself, alias both sides)
e = employees.alias("e")
m = employees.alias("m")
(e.join(m, F.col("e.manager_id") == F.col("m.employee_id"), "left")
    .select(F.col("e.name").alias("employee"),
            F.col("e.department"),
            F.col("m.name").alias("manager"))
    .orderBy("manager")
    .show())

## 11 · Union — stacking rows

`unionByName` stacks rows of two DataFrames with the same columns (matching by
name — safer than `union`, which matches by position).

In [ ]:
jan = orders.filter(F.month("order_ts") == 1)
feb = orders.filter(F.month("order_ts") == 2)
combined = jan.unionByName(feb)
print("jan:", jan.count(), "+ feb:", feb.count(), "-> ", combined.count())

## 12 · Window functions ⭐

Window functions compute across a set of rows **related to the current row**
without collapsing them. Build a window with `Window.partitionBy(...).orderBy(...)`,
then apply `row_number`, `rank`, `dense_rank`, `lag`/`lead`, or a running
aggregate.

In [ ]:
w = Window.partitionBy("department").orderBy(F.desc("salary"))
(employees
    .withColumn("rank_in_dept", F.row_number().over(w))
    .select("department","name","salary","rank_in_dept")
    .orderBy("department","rank_in_dept")
    .show())

In [ ]:
# LAG: compare each completed order to the customer's previous one
wc = Window.partitionBy("customer_id").orderBy("order_ts")
(orders.filter(F.col("status") == "completed")
    .withColumn("prev_amount", F.lag("amount").over(wc))
    .withColumn("change", F.col("amount") - F.lag("amount").over(wc))
    .select("customer_id","order_id","amount","prev_amount","change")
    .orderBy("customer_id","order_ts")
    .show())

In [ ]:
# Running total with an explicit frame (unbounded preceding -> current row)
wr = Window.orderBy("order_ts").rowsBetween(Window.unboundedPreceding, Window.currentRow)
(orders.filter(F.col("status") == "completed")
    .withColumn("running_total", F.round(F.sum("amount").over(wr), 2))
    .select("order_id","order_ts","amount","running_total")
    .show())

## 13 · String functions

`pyspark.sql.functions` has a rich text toolkit: `upper`/`lower`/`trim`,
`concat_ws`, `split`, `substring`, `regexp_extract`, `regexp_replace`, `length`.

In [ ]:
(customers.select(
    "name",
    F.upper(F.trim("country")).alias("country_uc"),
    F.split("name", " ").getItem(0).alias("first_name"),
    F.regexp_extract(F.coalesce("email", F.lit("")), "@(.+)$", 1).alias("email_domain"),
    F.concat_ws(" | ", "name", "country").alias("label"),
).show(truncate=False))

## 14 · Date & time functions

`to_date`, `date_trunc`, `datediff`, `year`/`month`/`dayofweek`, `date_format`,
`date_add`.

In [ ]:
(orders.select(
    "order_id", "order_ts",
    F.to_date("order_ts").alias("order_date"),
    F.date_trunc("month", "order_ts").alias("order_month"),
    F.year("order_ts").alias("yr"),
    F.date_format("order_ts", "yyyy-MM-dd").alias("formatted"),
    F.datediff(F.current_date(), F.col("order_ts")).alias("days_ago"),
).show(5))

## 15 · Complex types — arrays, structs, explode

Spark has first-class **arrays**, **structs** and **maps**. Aggregate values into
an array with `collect_list`, then flatten back with `explode` (one row per
element).

In [ ]:
baskets = order_items.groupBy("order_id").agg(
    F.collect_list("product_id").alias("product_ids"),
    F.size(F.collect_list("product_id")).alias("n_items"))
baskets.show(truncate=False)

# explode: one row per array element
baskets.select("order_id", F.explode("product_ids").alias("product_id")).show(6)

# struct + array literals
(orders.limit(2).select(
    "order_id",
    F.struct("status", "amount").alias("info"),
    F.array(F.lit("a"), F.lit("b")).alias("tags"),
).show(truncate=False))

## 16 · UDFs & pandas UDFs

When built-in functions can't express your logic, write a **UDF** (user-defined
function). Prefer built-ins when possible — UDFs are slower (they serialize data
to Python). **pandas UDFs** (vectorized) are much faster than row-at-a-time UDFs.

In [ ]:
from pyspark.sql.types import StringType

# Row-at-a-time UDF
@F.udf(StringType())
def band(amount):
    if amount is None: return "unknown"
    return "high" if amount >= 100 else "low"

orders.withColumn("band", band("amount")).select("order_id","amount","band").show(5)

In [ ]:
# Vectorized pandas UDF: operates on a whole pandas Series at once (fast)
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def with_tax(amount: pd.Series) -> pd.Series:
    return (amount * 1.2).round(2)

orders.withColumn("amount_with_tax", with_tax("amount")) \
      .select("order_id","amount","amount_with_tax").show(5)

## 17 · Spark SQL & temp views

Register a DataFrame as a **temporary view** and query it with SQL — mix and
match SQL and the DataFrame API freely (they compile to the same plan).

In [ ]:
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

spark.sql("""
    SELECT c.country, count(*) AS n, round(sum(o.amount), 2) AS revenue
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.country
    ORDER BY revenue DESC
""").show()

## 18 · Delta Lake — write, read, time travel, MERGE

On Databricks, tables are **Delta** by default — giving ACID transactions,
`UPDATE`/`DELETE`/`MERGE`, and **time travel**. Write a DataFrame as a table,
read it back, and see its version history.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS pyspark_bootcamp")

# Write a managed Delta table
(orders.write.format("delta").mode("overwrite")
    .saveAsTable("pyspark_bootcamp.orders_delta"))

# Read it back (spark.table or spark.read.table)
spark.table("pyspark_bootcamp.orders_delta").groupBy("status").count().show()

In [ ]:
# UPDATE / DELETE work on Delta tables
spark.sql("UPDATE pyspark_bootcamp.orders_delta SET amount = amount * 1.1 WHERE status = 'completed'")
spark.sql("DELETE FROM pyspark_bootcamp.orders_delta WHERE status = 'cancelled'")

# Time travel: every change is a versioned snapshot
spark.sql("DESCRIBE HISTORY pyspark_bootcamp.orders_delta").select("version","operation").show(truncate=False)

# Read an older version
v0 = spark.read.option("versionAsOf", 0).table("pyspark_bootcamp.orders_delta")
print("rows at version 0:", v0.count())

**`MERGE`** is the upsert — insert new rows, update changed ones in one atomic
step (the workhorse for change-data-capture):

```python
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "pyspark_bootcamp.orders_delta")
updates = spark.createDataFrame(
    [(1001, "completed", 999.0)], ["order_id", "status", "amount"])

(target.alias("t")
   .merge(updates.alias("s"), "t.order_id = s.order_id")
   .whenMatchedUpdate(set={"amount": "s.amount"})
   .whenNotMatchedInsertAll()
   .execute())
```

## 19 · Caching & performance

Because Spark is lazy, a DataFrame is recomputed every time you use it unless you
**cache** it. Other essentials: `explain` (see the plan), `repartition`/`coalesce`
(control parallelism), and **broadcast** joins (send a small table to every
executor to avoid a shuffle).

In [ ]:
completed = orders.filter(F.col("status") == "completed").cache()
print("count (materializes cache):", completed.count())   # action
print("reused from cache:", completed.agg(F.sum("amount")).first()[0])
completed.unpersist()

# Broadcast the small customers table in a join (avoids a shuffle)
joined = orders.join(F.broadcast(customers), "customer_id")
print("broadcast join rows:", joined.count())

# See the physical plan
orders.groupBy("status").count().explain()

**Performance tips:**
- Prefer built-in functions over UDFs; prefer **pandas UDFs** over plain UDFs.
- Filter and `select` **early** to shrink data before joins/aggregations.
- `broadcast()` the small side of a join with a large table.
- Mind **partitioning**: `repartition(n, "key")` for parallelism/shuffles,
  `coalesce(n)` to reduce partitions without a full shuffle.
- Adaptive Query Execution (AQE) auto-tunes shuffles on modern Databricks.
- **Actions** (`count`, `collect`, `write`) trigger work; chaining
  transformations is free until then.

## 20 · pandas interop

Move between Spark (distributed) and pandas (single machine), and use the
**pandas API on Spark** to run pandas-style code at Spark scale.

In [ ]:
# Spark -> pandas (only when the data fits on the driver!)
pdf = orders.limit(5).toPandas()
print(type(pdf).__name__, "\n", pdf[["order_id","amount"]])

# pandas -> Spark
sdf = spark.createDataFrame(pdf)
print("back to Spark:", sdf.count(), "rows")

# pandas API on Spark: same pandas syntax, distributed execution
import pyspark.pandas as ps
psdf = orders.pandas_api()               # or ps.DataFrame(orders)
print(psdf.groupby("status")["amount"].sum())

## 🎓 You've covered PySpark end to end

Spark's lazy model, building & inspecting DataFrames, `select`/`filter`/
`withColumn`, nulls, sorting, `groupBy`/`agg`, every join type, `union`, **window
functions**, string & date functions, complex types with `explode`, **UDFs** &
pandas UDFs, **Spark SQL**, **Delta Lake** (time travel + `MERGE`), caching &
performance, and pandas interop.

**Next:** point these at real tables in Unity Catalog, explore Structured
Streaming (`spark.readStream`), and use the Spark UI to inspect stages and
shuffles. The DataFrame API you learned here is exactly what production
Databricks pipelines are built from. Happy Sparking! 🚀